# 🚀 MLflow Demo - Multi-Environment Setup

This notebook demonstrates how to use MLflow for experiment tracking across:
- Google Colab (Web)
- Google Colab VSCode Extension
- Local MacOS

All experiments are logged to the same location via Google Drive sync.

## 1. Setup (Run in Colab)

In [ ]:
# Mount Google Drive (only needed in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Running locally")

In [ ]:
# Add project to path
import sys
if IN_COLAB:
    sys.path.insert(0, '/content/drive/MyDrive/repos/deep-learning')
else:
    # Local - adjust if running from different directory
    sys.path.insert(0, '..')

In [ ]:
# Install dependencies (Colab only)
if IN_COLAB:
    !pip install -q mlflow

## 2. Import Configuration

In [ ]:
from config.paths import (
    BRONZE, SILVER, GOLD,
    MLFLOW_TRACKING_URI,
    get_env_info
)

# Display environment info
env_info = get_env_info()
for key, value in env_info.items():
    print(f"{key}: {value}")

## 3. Configure MLflow

In [ ]:
import mlflow

# Set tracking URI to Google Drive location
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

# Create or get experiment
experiment_name = "demo-multi-env"
mlflow.set_experiment(experiment_name)
print(f"Experiment: {experiment_name}")

## 4. Run an Experiment

In [ ]:
import random
from datetime import datetime

# Simulate a training run
with mlflow.start_run(run_name=f"run_{datetime.now().strftime('%H%M%S')}"):
    # Log parameters
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("epochs", 10)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("environment", "colab" if IN_COLAB else "local")
    
    # Simulate training metrics
    for epoch in range(10):
        loss = 1.0 / (epoch + 1) + random.uniform(-0.05, 0.05)
        accuracy = 0.5 + (epoch * 0.05) + random.uniform(-0.02, 0.02)
        
        mlflow.log_metric("loss", loss, step=epoch)
        mlflow.log_metric("accuracy", accuracy, step=epoch)
    
    # Log final metrics
    mlflow.log_metric("final_accuracy", accuracy)
    
    print(f"✅ Run logged! Final accuracy: {accuracy:.4f}")
    print(f"View in MLflow UI or at: {MLFLOW_TRACKING_URI}")

## 5. View Logged Runs

In [ ]:
# List recent runs
runs = mlflow.search_runs(experiment_names=[experiment_name], max_results=5)
if not runs.empty:
    print("Recent runs:")
    print(runs[["run_id", "params.environment", "metrics.final_accuracy", "start_time"]].to_string())
else:
    print("No runs found yet.")

## 6. View in MLflow UI (Local Only)

Run this in your Mac terminal:

```bash
cd "/Users/khangnghiem/Library/CloudStorage/GoogleDrive-khangnghiem@gmail.com/My Drive/data_lake/experiments"
mlflow ui --backend-store-uri file://./mlruns --port 5000
```

Then open http://localhost:5000